# Context-Aware Chatbot Using RAG

## 1. Problem Statement
The goal of this project is to build a chatbot that can remember context and retrieve relevant external information from a custom knowledge base.

## 2. Dataset / Knowledge Base

## 3. Document Vectorization

## 4. Retrieval Mechanism

## 5. Question Answering

## 6. Context Memory

## 7. Streamlit Deployment

## 8. Final Summary

In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

C:\Users\ayesh\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
documents = [
    "Python is a high-level programming language used for web development, machine learning, automation, and data analysis.",
    "Machine learning is a branch of artificial intelligence that allows systems to learn patterns from data and make predictions.",
    "Streamlit is an open-source Python framework used to build interactive web apps for data science and machine learning projects.",
    "Natural language processing, or NLP, is a field of AI that focuses on the interaction between computers and human language.",
    "Retrieval-Augmented Generation, or RAG, combines information retrieval with language generation to answer questions using external knowledge.",
    "Scikit-learn is a Python machine learning library that provides tools for classification, regression, clustering, and preprocessing.",
    "Transformers are deep learning models that are widely used for NLP tasks such as text classification, translation, summarization, and question answering.",
    "A vector store is used to save numerical representations of documents so that similar content can be retrieved efficiently.",
    "Cosine similarity measures how similar two text vectors are and is commonly used in information retrieval systems.",
    "Gradio and Streamlit are popular tools for deploying machine learning applications with simple user interfaces."
]

In [4]:
df = pd.DataFrame({"document": documents})
df

,document
0,Python is a high-level programming language us...
1,Machine learning is a branch of artificial int...
2,Streamlit is an open-source Python framework u...
3,"Natural language processing, or NLP, is a fiel..."
4,"Retrieval-Augmented Generation, or RAG, combin..."
5,Scikit-learn is a Python machine learning libr...
6,Transformers are deep learning models that are...
7,A vector store is used to save numerical repre...
8,Cosine similarity measures how similar two tex...
9,Gradio and Streamlit are popular tools for dep...


In [5]:
vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(df["document"])

In [6]:
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

C:\Users\ayesh\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ayesh\.cache\huggingface\hub\models--distilbert-base-cased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Device set to use cpu


In [7]:
def retrieve_documents(query, top_k=3):
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, document_vectors).flatten()
    top_indices = similarities.argsort()[::-1][:top_k]
    
    retrieved_docs = df.iloc[top_indices]["document"].tolist()
    return retrieved_docs

In [8]:
def answer_question(query, chat_history=None, top_k=3):
    retrieved_docs = retrieve_documents(query, top_k=top_k)
    context = " ".join(retrieved_docs)
    
    result = qa_pipeline(question=query, context=context)
    
    answer = result["answer"]
    score = result["score"]
    
    return {
        "question": query,
        "answer": answer,
        "confidence": score,
        "retrieved_docs": retrieved_docs
    }

In [9]:
response = answer_question("What is Streamlit?")
response

{'question': 'What is Streamlit?',
 'answer': 'an open-source Python framework',
 'confidence': 0.45248472690582275,
 'retrieved_docs': ['Gradio and Streamlit are popular tools for deploying machine learning applications with simple user interfaces.',
  'Streamlit is an open-source Python framework used to build interactive web apps for data science and machine learning projects.',
  'Cosine similarity measures how similar two text vectors are and is commonly used in information retrieval systems.']}

In [10]:
questions = [
    "What is machine learning?",
    "What is RAG?",
    "What is Streamlit used for?",
    "What does cosine similarity do?"
]

for q in questions:
    result = answer_question(q)
    print("Question:", result["question"])
    print("Answer:", result["answer"])
    print("Confidence:", round(result["confidence"], 4))
    print("Retrieved Documents:", result["retrieved_docs"])
    print("-" * 80)

Question: What is machine learning?
Answer: a Python machine learning library
Confidence: 0.309
Retrieved Documents: ['Gradio and Streamlit are popular tools for deploying machine learning applications with simple user interfaces.', 'Scikit-learn is a Python machine learning library that provides tools for classification, regression, clustering, and preprocessing.', 'Python is a high-level programming language used for web development, machine learning, automation, and data analysis.']
--------------------------------------------------------------------------------
Question: What is RAG?
Answer: Retrieval-Augmented Generation
Confidence: 0.3603
Retrieved Documents: ['Retrieval-Augmented Generation, or RAG, combines information retrieval with language generation to answer questions using external knowledge.', 'Gradio and Streamlit are popular tools for deploying machine learning applications with simple user interfaces.', 'Cosine similarity measures how similar two text vectors are and 

In [11]:
chat_history = []

def chatbot_with_memory(query):
    global chat_history
    
    result = answer_question(query, chat_history=chat_history, top_k=3)
    
    chat_history.append({
        "user": query,
        "bot": result["answer"]
    })
    
    return result

In [12]:
print(chatbot_with_memory("What is Python?"))
print(chatbot_with_memory("What is it used for?"))
print(chat_history)

{'question': 'What is Python?', 'answer': 'a high-level programming language', 'confidence': 0.4593736231327057, 'retrieved_docs': ['Scikit-learn is a Python machine learning library that provides tools for classification, regression, clustering, and preprocessing.', 'Python is a high-level programming language used for web development, machine learning, automation, and data analysis.', 'Streamlit is an open-source Python framework used to build interactive web apps for data science and machine learning projects.']}
{'question': 'What is it used for?', 'answer': 'information retrieval systems', 'confidence': 0.6090369820594788, 'retrieved_docs': ['Cosine similarity measures how similar two text vectors are and is commonly used in information retrieval systems.', 'Python is a high-level programming language used for web development, machine learning, automation, and data analysis.', 'A vector store is used to save numerical representations of documents so that similar content can be ret

In [13]:
def chatbot_with_context(query, memory_limit=3):
    global chat_history
    
    recent_memory = " ".join(
        [f"User: {item['user']} Bot: {item['bot']}" for item in chat_history[-memory_limit:]]
    )
    
    enhanced_query = recent_memory + " " + query if recent_memory else query
    
    result = answer_question(enhanced_query, top_k=3)
    
    chat_history.append({
        "user": query,
        "bot": result["answer"]
    })
    
    return result

In [14]:
chat_history = []

print(chatbot_with_context("What is Python?"))
print(chatbot_with_context("What is it used for?"))

{'question': 'What is Python?', 'answer': 'a high-level programming language', 'confidence': 0.4593736231327057, 'retrieved_docs': ['Scikit-learn is a Python machine learning library that provides tools for classification, regression, clustering, and preprocessing.', 'Python is a high-level programming language used for web development, machine learning, automation, and data analysis.', 'Streamlit is an open-source Python framework used to build interactive web apps for data science and machine learning projects.']}
{'question': 'User: What is Python? Bot: a high-level programming language What is it used for?', 'answer': 'web development, machine learning, automation, and data analysis', 'confidence': 0.7398647665977478, 'retrieved_docs': ['Python is a high-level programming language used for web development, machine learning, automation, and data analysis.', 'Natural language processing, or NLP, is a field of AI that focuses on the interaction between computers and human language.', 

In [15]:
df.to_csv("knowledge_base.csv", index=False)
print("Knowledge base saved as knowledge_base.csv")

Knowledge base saved as knowledge_base.csv


In [17]:
print("""
Summary:
- Built a context-aware chatbot using Retrieval-Augmented Generation (RAG).
- Created a custom knowledge base for answering user queries.
- Used TF-IDF vectorization and cosine similarity for document retrieval.
- Used a lightweight Hugging Face question-answering model for answer generation.
- Implemented context memory using chat history.
- Deployed the chatbot with Streamlit.
""")


Summary:
- Built a context-aware chatbot using Retrieval-Augmented Generation (RAG).
- Created a custom knowledge base for answering user queries.
- Used TF-IDF vectorization and cosine similarity for document retrieval.
- Used a lightweight Hugging Face question-answering model for answer generation.
- Implemented context memory using chat history.
- Deployed the chatbot with Streamlit.

